In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit193_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit488_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit224_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit396_aug_0.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit492_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit293_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit344_aug_0.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit2_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit180_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_dete

In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall

In [2]:
import time
import json
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

# =============================
# CONFIG
# =============================
DATA_PATH = "/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml"
RESULTS_CSV = "yolov8s_ablation_results.csv"

# Global Best Parameters (update as you go)
BEST_PARAMS = {
    "epochs": 32,          # default, update after finding best
    "batch": 32,
    "lr0": 0.01,
    "weight_decay": 0.0005,
    "optimizer": "SGD",
    "img_size": 640,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# Initialize results DataFrame
if Path(RESULTS_CSV).exists():
    results_df = pd.read_csv(RESULTS_CSV)
else:
    results_df = pd.DataFrame(columns=[
        "Case", "Exp No", "Experiment Name", "Epochs", "Batch", "lr0", "Weight Decay",
        "Optimizer", "Image Size", "Dropout", "Freeze Layers", "Activation",
        "Training Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ])


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def run_experiment(case, exp_no, exp_name, epochs=32, batch=32, lr0=0.01,
                   weight_decay=0.0005, optimizer="SGD", img_size=640,
                   dropout=0.0, freeze=0, activation="SiLU"):
    global results_df

    model = YOLO("yolov8m.pt")  # Using YOLOv8 samll

    # Track training time
    start_time = time.time()
    model.train(
        data=DATA_PATH,
        epochs=epochs,
        batch=batch,
        lr0=lr0,
        optimizer=optimizer,
        imgsz=img_size,
        dropout=dropout,
        freeze=freeze,
        verbose=False
    )
    train_time = time.time() - start_time

    # ✅ FIX: Load YOLO metrics from results.csv
    results_file = Path(model.trainer.save_dir) / "results.csv"
    results_df_yolo = pd.read_csv(results_file)
    last_row = results_df_yolo.iloc[-1]

    precision = last_row["metrics/precision(B)"]
    recall = last_row["metrics/recall(B)"]
    map50 = last_row["metrics/mAP50(B)"]
    map5095 = last_row["metrics/mAP50-95(B)"]

    new_row = {
        "Case": case,
        "Exp No": exp_no,
        "Experiment Name": exp_name,
        "Epochs": epochs,
        "Batch": batch,
        "lr0": lr0,
        "Weight Decay": weight_decay,
        "Optimizer": optimizer,
        "Image Size": img_size,
        "Dropout": dropout,
        "Freeze Layers": freeze,
        "Activation": activation,
        "Training Time (s)": round(train_time, 2),
        "Precision": precision,
        "Recall": recall,
        "mAP@0.5": map50,
        "mAP@0.5:0.95": map5095,
    }
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

    results_df.to_csv(RESULTS_CSV, index=False)
    display(results_df.tail(10))
# =============================
# Wrapper to use BEST_PARAMS
# =============================
def run_with_best(case, exp_no, exp_name, **kwargs):
    params = BEST_PARAMS.copy()
    params.update(kwargs)
    run_experiment(case, exp_no, exp_name, **params)

In [5]:
# Case 1: Epoch Variation
def case1_epoch_variation():
    run_with_best("Case 1: Epoch Variation", 1, "epochs_16", epochs=16)
    run_with_best("Case 1: Epoch Variation", 2, "epochs_32", epochs=32)
    run_with_best("Case 1: Epoch Variation", 3, "epochs_64", epochs=64)

# Case 2: Weight Decay
def case2_weight_decay():
    run_with_best("Case 2: Weight Decay", 1, "wd_0", weight_decay=0)
    run_with_best("Case 2: Weight Decay", 2, "wd_0.0001", weight_decay=0.0001)
    run_with_best("Case 2: Weight Decay", 3, "wd_0.0005", weight_decay=0.0005)
    run_with_best("Case 2: Weight Decay", 4, "wd_0.001", weight_decay=0.001)
    run_with_best("Case 2: Weight Decay", 5, "wd_0.005", weight_decay=0.005)

# Case 3: Batch Size
def case3_batch_size():
    run_with_best("Case 3: Batch Size", 1, "batch_16", batch=16)
    run_with_best("Case 3: Batch Size", 2, "batch_32", batch=32)
    run_with_best("Case 3: Batch Size", 3, "batch_64", batch=64)  # may OOM

# Case 4: Dropout
def case4_dropout():
    for i, d in enumerate([0, 0.1, 0.2, 0.3, 0.5], start=1):
        run_with_best("Case 4: Dropout", i, f"dropout_{d}", dropout=d)

# Case 6: Optimizer
def case6_optimizer():
    opts = ["SGD", "Adam", "AdamW", "RMSProp", "Adamax"]
    for i, opt in enumerate(opts, start=1):
        run_with_best("Case 6: Optimizer", i, f"optimizer_{opt}", optimizer=opt)
# Case 7: Learning Rate
def case7_learning_rate():
    for i, lr in enumerate([0.001, 0.005, 0.01, 0.05], start=1):
        run_with_best("Case 7: Learning Rate", i, f"lr_{lr}", lr0=lr)

# Case 8: Freeze Layers
def case8_freeze_layers():
    for i, freeze in enumerate([0, 10, 15, 22], start=1):
        run_with_best("Case 8: Freeze Layers", i, f"freeze_{freeze}", freeze=freeze)
# Case 10: Image Size
def case10_image_size():
    sizes = [320, 512, 640]
    for i, sz in enumerate(sizes, start=1):
        run_with_best("Case 10: Image Size", i, f"imgsz_{sz}", img_size=sz)

In [5]:
case1_epoch_variation()

Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.997      0.994      0.994      0.741
Speed: 0.1ms preprocess, 7.4ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/train


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,763.86,0.99663,0.99388,0.99412,0.74142


Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,763.86,0.99663,0.99388,0.99412,0.74142
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,1474.82,0.99606,0.99778,0.99467,0.76455


Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=64, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.997      0.996      0.995      0.791
Speed: 0.1ms preprocess, 7.6ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/train3


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,763.86,0.99663,0.99388,0.99412,0.74142
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,1474.82,0.99606,0.99778,0.99467,0.76455
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,2931.67,0.99653,0.99555,0.99452,0.79041


In [6]:
case3_batch_size()

Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.997      0.995      0.764
Speed: 0.1ms preprocess, 7.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/train


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0005,SGD,640,0.0,0,SiLU,1637.85,0.99598,0.99666,0.9948,0.7645


Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.998      0.995      0.764
Speed: 0.1ms preprocess, 7.4ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to runs/detect/train2


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0005,SGD,640,0.0,0,SiLU,1637.85,0.99598,0.99666,0.99480,0.76450
1,Case 3: Batch Size,2,batch_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,1477.32,0.99606,0.99778,0.99467,0.76455


Ultralytics 8.3.191 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

OutOfMemoryError: CUDA out of memory. Tried to allocate 600.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 67.12 MiB is free. Process 3978 has 15.82 GiB memory in use. Of the allocated memory 15.02 GiB is allocated by PyTorch, and 484.71 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)